# TnDrr AI/ML Internship Assignment - Task 3: Local LLM Optimization & Benchmarking

**Objective**: Compare FP16 (unquantized) versus 4-bit Quantization on the same 10 standardized evaluation prompts.
Measure tokens per second, latency, RAM, and VRAM.

In [ ]:
# 1. Environment & GPU Verification
!nvidia-smi
!pip install -q torch transformers accelerate bitsandbytes psutil matplotlib pandas

In [ ]:
# 2. Standardized Prompts
PROMPTS = [
    {"id": "p01", "category": "Summarization", "prompt": "Summarize in one sentence: Artificial intelligence models trained on massive multimodal datasets have achieved unprecedented performance across vision and language, leading to rapid enterprise adoption."},
    {"id": "p02", "category": "Code Generation", "prompt": "Write a Python function `binary_search(arr, target)` with type hints that returns the index of target or -1."},
    {"id": "p03", "category": "Math Reasoning", "prompt": "A train travels at 60 km/h for 2.5 hours, then at 80 km/h for 1.5 hours. What is the total distance and average speed?"},
    {"id": "p04", "category": "Information Extraction", "prompt": "Extract company name, price, and target: 'In October 2024, Cyberdyne Systems acquired Omni Robotics for $450 million in cash.'"},
    {"id": "p05", "category": "Logic", "prompt": "Alice is taller than Bob. Charlie is shorter than Alice but taller than Bob. David is taller than Charlie but shorter than Alice. Order them from tallest to shortest."},
    {"id": "p06", "category": "Instruction Adherence", "prompt": "You are a Linux sysadmin. Explain in exactly three bullet points what `chmod 755 script.sh` does."},
    {"id": "p07", "category": "Technical Explanation", "prompt": "Explain the difference between Transformer Key, Query, and Value matrices using an intuitive analogy."},
    {"id": "p08", "category": "Creative Writing", "prompt": "Write a haiku about compiling machine learning models onto edge hardware."},
    {"id": "p09", "category": "Few-shot Classification", "prompt": "Classify sentiment as Positive or Negative:\nReview: 'Battery lasts all day.' Sentiment: Positive\nReview: 'Lags frequently.' Sentiment: Negative\nReview: 'Setup took 5 minutes.' Sentiment:"},
    {"id": "p10", "category": "Factual Retrieval", "prompt": "What is the capital of Australia, and what is the primary river flowing through it?"}
]
print(f"Loaded {len(PROMPTS)} standardized benchmark prompts.")

In [ ]:
# 3. Benchmark Function
import time, os, psutil, torch, json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

def benchmark_hf_model(model, label="FP16"):
    results = []
    total_tokens, total_time = 0, 0.0
    
    print(f"\n=== Running Benchmark: {label} ===")
    for p in PROMPTS:
        inputs = tokenizer(p["prompt"], return_tensors="pt").to(model.device)
        prompt_len = inputs["input_ids"].shape[1]
        
        torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=128,
                temperature=0.0,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        elapsed = time.perf_counter() - t0
        gen_tokens = out[0][prompt_len:]
        num_gen = len(gen_tokens)
        tok_sec = num_gen / max(elapsed, 1e-6)
        
        peak_vram = torch.cuda.max_memory_allocated() / (1024 * 1024) if torch.cuda.is_available() else 0.0
        ram_mb = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
        text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        
        total_tokens += num_gen
        total_time += elapsed
        
        results.append({
            "prompt_id": p["id"],
            "category": p["category"],
            "prompt": p["prompt"],
            "response": text,
            "tokens": num_gen,
            "latency_sec": round(elapsed, 3),
            "tokens_per_sec": round(tok_sec, 2),
            "peak_vram_mb": round(peak_vram, 2),
            "ram_mb": round(ram_mb, 2)
        })
        print(f"[{p['id']}]: {num_gen} tokens in {elapsed:.2f}s ({tok_sec:.1f} tok/s) | VRAM: {peak_vram:.1f} MB")
        
    avg_speed = total_tokens / max(total_time, 1e-6)
    return {
        "quantization": label,
        "avg_speed_tok_s": round(avg_speed, 2),
        "total_tokens": total_tokens,
        "records": results
    }

In [ ]:
# 4. Benchmark Setting 1: FP16
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
res_fp16 = benchmark_hf_model(model_fp16, label="FP16")
del model_fp16
torch.cuda.empty_cache()

In [ ]:
# 5. Benchmark Setting 2: 4-bit (NF4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)
model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
res_4bit = benchmark_hf_model(model_4bit, label="4-bit NF4")
del model_4bit
torch.cuda.empty_cache()

In [ ]:
# 6. Results Comparison
print("=" * 60)
print(f"FP16 Average Decode Speed : {res_fp16['avg_speed_tok_s']} tokens/sec")
print(f"4-bit Average Decode Speed: {res_4bit['avg_speed_tok_s']} tokens/sec")
print("=" * 60)

final_bench = {
    "fp16": res_fp16,
    "int4": res_4bit
}
with open("benchmark_results.json", "w") as f:
    json.dump(final_bench, f, indent=2)
print("Saved benchmark_results.json")

In [ ]:
# 7. Download Results
from google.colab import files
files.download("benchmark_results.json")